<a href="https://colab.research.google.com/github/vitorjensen/monitor-trading-margens/blob/main/notebooks/01_analise_exploratoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Projeto prático 01** - Monitor de Margens e Trading

### **Objetivo:** Descobrir em qual dia e para qual produto a margem de lucro da mesa de Trading colapsou, tornando inviável a venda daquele combustível/insumo no mercado spot.

##### Configuração do PlayWright para Web Scraper automáticos de dados

In [ ]:
# 1. Instala as bibliotecas do Python
!pip install playwright pandas openpyxl

# 2. Baixa os navegadores do Playwright
!playwright install

# 3. Instala as dependências de sistema do Linux para o Chromium rodar na nuvem
!playwright install-deps

In [2]:
import os

# Garante que as pastas do nosso repositório existam no menu lateral do Colab
os.makedirs('src', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
print("📂 Estrutura de pastas verificada com sucesso!")

📂 Estrutura de pastas verificada com sucesso!


##### Escrevendo o código Web Scraper no caminho: src/scraper.py

In [9]:
%%writefile src/scraper.py
from playwright.sync_api import sync_playwright
import os

def executar_scraper():
    print("🚀 [COLAB] Iniciando o robô de extração em segundo plano...")

    with sync_playwright() as p:
        # IMPORTANTE: headless=True é obrigatório na nuvem do Colab
        browser = p.chromium.launch(headless=True) # Armazenando o browser numa variávelchamada: "Browser".
        page = browser.new_page() # Fazer com que o browser abra uma nova página

        # Teste inicial acessando um site simples
        print("🌍 Navegando até o site de teste...")
        page.goto('https://www.uol.com.br/', timeout=0) # Função "goto" para apontar o site que desejamos abrir na página.

        # Capturando o título da página
        texto_titulo = page.locator('h1').text_content()

        print("✅ Raspagem realizada com sucesso!")
        print(f"📄 Título capturado da página: {texto_titulo.strip()}")

        browser.close()

if __name__ == "__main__":
    executar_scraper()

Overwriting src/scraper.py


##### Teste do Web Scraper com um site teste

In [10]:
!python src/scraper.py

🚀 [COLAB] Iniciando o robô de extração em segundo plano...
🌍 Navegando até o site de teste...
✅ Raspagem realizada com sucesso!
📄 Título capturado da página: UOL - Seu universo online


####Configuração para criar a base de dados de teste utilizando os seeds

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:

# Configuração para criar dados de testes idênticos
np.random.seed(42)
datas = pd.date_range(start='2026-06-01', end='2026-06-15', freq='D')
produtos = ['Etanol Hidratado', 'Lubrificante Hidráulico']

# Gerando combinações de data e produto
lista_comb = [(d, p) for d in datas for p in produtos]
df_base = pd.DataFrame(lista_comb, columns=['Data', 'Produto'])

# --- DATAFRAME 1: CUSTOS INTERNOS DA INDÚSTRIA ---
df_custos = df_base.copy()
df_custos['Custo_Materia_Prima_Litro'] = np.where(df_custos['Produto'] == 'Etanol Hidratado', np.random.uniform(2.10, 2.60, len(df_base)), np.random.uniform(12.00, 15.50, len(df_base)))
df_custos['Custo_Operacional_Litro'] = np.where(df_custos['Produto'] == 'Etanol Hidratado', np.random.uniform(0.40, 0.60, len(df_base)), np.random.uniform(3.00, 4.50, len(df_base)))
df_custos = df_custos.sample(frac=1).reset_index(drop=True)

# --- DATAFRAME 2: PREÇOS DO MERCADO SPOT (TRADING) ---
df_mercado = df_base.copy()
df_mercado['Preco_Venda_Spot_Litro'] = np.where(df_mercado['Produto'] == 'Etanol Hidratado', np.random.uniform(2.80, 3.40, len(df_base)), np.random.uniform(16.00, 21.00, len(df_base)))

# Injetando a anomalia oculta para você caçar na análise!
df_mercado.loc[(df_mercado['Data'] == '2026-06-10') & (df_mercado['Produto'] == 'Lubrificante Hidráulico'), 'Preco_Venda_Spot_Litro'] = 13.50
df_mercado = df_mercado.sample(frac=1).reset_index(drop=True)

# --- EXPORTANDO PARA AS PASTAS DO REPOSITÓRIO ---
# Criando caminhos relativos para funcionar direto na sua pasta local do Mac
os.makedirs('data/raw', exist_ok=True)

df_custos.to_csv('data/raw/custos_industriais.csv', index=False)
df_mercado.to_csv('data/raw/precos_mercado.csv', index=False)

print("✅ Arquivos gerados e salvos com sucesso em:")
print("   -> data/raw/custos_industriais.csv")
print("   -> data/raw/precos_mercado.csv")

✅ Arquivos gerados e salvos com sucesso em:
   -> data/raw/custos_industriais.csv
   -> data/raw/precos_mercado.csv


#### **Passo 1:** Consolidação do **Custo Total** Interno: Saber quanto custa, de fato, produzir cada litro de produto na fábrica.

In [ ]:
df_custos_industriais = pd.read_csv('data/raw/custos_industriais.csv')
df_custos_industriais.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616


In [ ]:
df_custos_industriais['Custo_Total_Litro'] = df_custos_industriais['Custo_Materia_Prima_Litro'] + df_custos_industriais['Custo_Operacional_Litro']
df_custos_industriais.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735


#### **Passo 2:** Mesclar as planilhas para sabermos o preço de mercado e o custo juntos

In [ ]:
df_precos_mercado = pd.read_csv('data/raw/precos_mercado.csv')
df_precos_mercado.head()

,Data,Produto,Preco_Venda_Spot_Litro
0,2026-06-05,Lubrificante Hidráulico,20.683650
1,2026-06-13,Lubrificante Hidráulico,17.695149
2,2026-06-03,Etanol Hidratado,2.980527
3,2026-06-06,Lubrificante Hidráulico,17.705332
4,2026-06-04,Etanol Hidratado,2.822132


In [ ]:
df_merge = pd.merge(df_custos_industriais, df_precos_mercado, on=['Data', 'Produto'])
df_merge.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284,3.093672
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702,19.387822
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735,17.705332


##### Definir quais produtos estão com um Custo Total MAIOR que um Preço de Venda

In [ ]:
df_custo_maior = df_merge[df_merge['Custo_Total_Litro'] > df_merge['Preco_Venda_Spot_Litro']]
df_custo_maior

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro,Margem_Absoluta_Litro
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514,-1.489283
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465,-0.602550
6,2026-06-08,Lubrificante Hidráulico,14.318828,3.373938,17.692766,17.289708,-0.403058
10,2026-06-15,Etanol Hidratado,2.396207,0.577443,2.973650,2.911911,-0.061739
19,2026-06-02,Etanol Hidratado,2.465997,0.565748,3.031744,2.951069,-0.080675
22,2026-06-10,Lubrificante Hidráulico,12.646991,3.115470,15.762460,13.500000,-2.262460
29,2026-06-13,Lubrificante Hidráulico,15.226560,4.307191,19.533751,17.695149,-1.838602


#### **Passo 3:** Calcular a Margem de Arbitragem: Descobrir a sobra financeira de cada operação de Trading

In [ ]:
df_merge['Margem_Absoluta_Litro'] = df_merge['Preco_Venda_Spot_Litro'] - df_merge['Custo_Total_Litro']
df_merge.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro,Margem_Absoluta_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284,3.093672,0.361388
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702,19.387822,2.721120
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514,-1.489283
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465,-0.602550
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735,17.705332,0.017597


In [ ]:
df_merge['Margem_Percentual'] = (df_merge['Margem_Absoluta_Litro'] / df_merge['Preco_Venda_Spot_Litro']) * 100
df_merge.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro,Margem_Absoluta_Litro,Margem_Percentual
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284,3.093672,0.361388,11.681515
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702,19.387822,2.721120,14.035203
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514,-1.489283,-9.044862
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465,-0.602550,-3.246416
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735,17.705332,0.017597,0.099388


#### **Passo 4:** Diagnóstico Final: Onde está a Crise?

##### Achando os produtos que ficaram abaixo do nível aceitável

In [ ]:
df_produtos_abaixo_do_nivel = df_merge[df_merge['Margem_Percentual'] < 0].sort_values(by='Margem_Percentual', ascending=True)
df_produtos_abaixo_do_nivel

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro,Margem_Absoluta_Litro,Margem_Percentual
22,2026-06-10,Lubrificante Hidráulico,12.646991,3.115470,15.762460,13.500000,-2.262460,-16.758966
29,2026-06-13,Lubrificante Hidráulico,15.226560,4.307191,19.533751,17.695149,-1.838602,-10.390428
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514,-1.489283,-9.044862
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465,-0.602550,-3.246416
19,2026-06-02,Etanol Hidratado,2.465997,0.565748,3.031744,2.951069,-0.080675,-2.733758
6,2026-06-08,Lubrificante Hidráulico,14.318828,3.373938,17.692766,17.289708,-0.403058,-2.331203
10,2026-06-15,Etanol Hidratado,2.396207,0.577443,2.973650,2.911911,-0.061739,-2.120213


##### Calculando os produtos que tiveram pior performance

In [ ]:
df_produtos_pior_performance = df_merge.groupby('Produto')['Margem_Percentual'].mean().reset_index()
df_produtos_pior_performance

,Produto,Margem_Percentual
0,Etanol Hidratado,9.043627
1,Lubrificante Hidráulico,4.866700
